In [ ]:
# CELL 0 — Montage Google Drive (persistance cache)
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
from pathlib import Path

# Dossier persistant sur Drive
DRIVE_CACHE = Path("/content/drive/MyDrive/AFND_cache")
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)

LOCAL_CACHE = Path("/content/AFND_cache")
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

# Si cache existe sur Drive → copier en local pour accès rapide
cache_file_drive = DRIVE_CACHE / "sentiment_scores.pkl"
cache_file_local = LOCAL_CACHE / "sentiment_scores.pkl"

if cache_file_drive.exists() and not cache_file_local.exists():
    shutil.copy(cache_file_drive, cache_file_local)
    print(f"✅ Cache copié depuis Drive → local ({cache_file_local.stat().st_size/1e6:.1f} MB)")
elif cache_file_local.exists():
    print("✅ Cache déjà en local.")
else:
    print("⚠️  Aucun cache trouvé — Cell 6 va recalculer (~8h).")

print(f"   Drive cache : {cache_file_drive}")
print(f"   Local cache : {cache_file_local}")

Mounted at /content/drive
⚠️  Aucun cache trouvé — Cell 6 va recalculer (~8h).
   Drive cache : /content/drive/MyDrive/AFND_cache/sentiment_scores.pkl
   Local cache : /content/AFND_cache/sentiment_scores.pkl


In [ ]:
# ============================================================
# SECTION 0 — INSTALLATION ROBUSTE
# ============================================================
import subprocess, sys

def pip_install(pkg, extra_args=None):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q']
    if extra_args:
        cmd.extend(extra_args)
    cmd.append(pkg)
    try:
        subprocess.check_call(cmd)
        print(f"  ✓ {pkg}")
        return True
    except subprocess.CalledProcessError as e:
        print(f"  ✗ {pkg} — erreur : {e}")
        return False

print("Upgrade pip...")
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q',
     '--upgrade', 'pip']
)

print("\nInstallation des packages...")
PACKAGES = [
    'transformers>=4.35.0',
    'datasets>=2.14.0',
    'tokenizers>=0.15.0',
    'sentencepiece>=0.1.99',
    'torch>=2.0.0',
    'scikit-learn>=1.3.0',
    'scipy>=1.11.0',
    'pandas>=2.0.0',
    'numpy>=1.24.0',
    'matplotlib>=3.7.0',
    'seaborn>=0.12.0',
    'nltk>=3.8.0',
    'pyarabic>=0.6.15',
    'mlxtend>=0.22.0',
    'gdown>=4.7.0',
    'accelerate>=0.24.0',
]

failed = []
for pkg in PACKAGES:
    ok = pip_install(pkg)
    if not ok:
        pkg_name = pkg.split('>=')[0].split('==')[0]
        print(f"    → Tentative sans contrainte : {pkg_name}")
        ok2 = pip_install(pkg_name)
        if not ok2:
            failed.append(pkg_name)

if failed:
    print(f"\n⚠ Packages non installés : {failed}")
else:
    print("\n✓ Tous les packages installés.")

print("\nVersions installées :")
checks = [
    ('torch',        'torch'),
    ('transformers', 'transformers'),
    ('sklearn',      'scikit-learn'),
    ('pandas',       'pandas'),
    ('numpy',        'numpy'),
]
for mod, name in checks:
    try:
        m = __import__(mod)
        v = getattr(m, '__version__', 'unknown')
        print(f"  {name:15s} : {v}")
    except ImportError:
        print(f"  {name:15s} : ✗ non disponible")

# ============================================================
# SECTION 1 — IMPORTS GLOBAUX
# ============================================================
print("\nChargement des imports...")

import os, json, re, zipfile, random, warnings, pickle
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix
)
from sklearn.calibration import calibration_curve

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
    pipeline as hf_pipeline
)

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)

try:
    from mlxtend.evaluate import mcnemar, mcnemar_table
    MLXTEND_OK = True
    print("✓ mlxtend chargé.")
except ImportError:
    MLXTEND_OK = False
    print("⚠ mlxtend indisponible — McNemar désactivé.")

try:
    import gdown
    GDOWN_OK = True
    print("✓ gdown chargé.")
except ImportError:
    GDOWN_OK = False
    print("⚠ gdown indisponible.")

warnings.filterwarnings('ignore')

# ── Reproductibilité ─────────────────────────────────────────
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n✓ Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ── Colab detection ──────────────────────────────────────────
IS_COLAB = 'google.colab' in sys.modules
DL_KWARGS = dict(
    num_workers=0 if IS_COLAB else 2,   # ← fix deadlock Colab
    pin_memory=torch.cuda.is_available()
)

arabic_stopwords = set(stopwords.words('arabic'))
print("✓ Imports terminés.")

# ============================================================
# SECTION 2 — TÉLÉCHARGEMENT AFND DEPUIS GOOGLE DRIVE
# ============================================================
FILE_ID    = '13AN1hNG3BrLcNXdv2CZj0EZ0SNNXqR4i'
ZIP_PATH   = '/content/AFND.zip'
EXTRACT_TO = '/content/AFND_extracted'
OUTPUT_DIR = '/content/AFND_results'
CACHE_DIR  = '/content/AFND_cache'

os.makedirs(EXTRACT_TO, exist_ok=True)
os.makedirs(OUTPUT_DIR,  exist_ok=True)
os.makedirs(CACHE_DIR,   exist_ok=True)

# ── Cache helpers ─────────────────────────────────────────────
def cache_save(obj, name):
    path = os.path.join(CACHE_DIR, f'{name}.pkl')
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    print(f"  💾 Cache sauvegardé : {name}")

def cache_load(name):
    path = os.path.join(CACHE_DIR, f'{name}.pkl')
    if os.path.exists(path):
        with open(path, 'rb') as f:
            print(f"  📂 Cache chargé : {name}")
            return pickle.load(f)
    return None

# ── Téléchargement ───────────────────────────────────────────
def download_afnd(file_id, zip_path):
    url = f'https://drive.google.com/uc?id={file_id}'
    print(f"Téléchargement depuis Drive (id={file_id})...")

    # Méthode 1 : gdown
    if GDOWN_OK:
        try:
            gdown.download(url, zip_path, quiet=False, fuzzy=True)
            if (os.path.exists(zip_path)
                    and os.path.getsize(zip_path) > 1e6):
                print(f"✓ gdown OK "
                      f"({os.path.getsize(zip_path)/1e6:.1f} MB)")
                return True
        except Exception as e:
            print(f"  gdown echec : {e}")

    # Méthode 2 : gdown exécutable (corrigé — pas -m)
    try:
        subprocess.check_call([
            'gdown', '--fuzzy',
            f'https://drive.google.com/file/d/{file_id}/view',
            '-O', zip_path
        ])
        if (os.path.exists(zip_path)
                and os.path.getsize(zip_path) > 1e6):
            print("✓ gdown subprocess OK")
            return True
    except Exception as e:
        print(f"  gdown subprocess echec : {e}")

    # Méthode 3 : wget
    print("  Tentative wget...")
    try:
        cmd = (
            f'wget -q --show-progress '
            f'"https://drive.google.com/uc?export=download'
            f'&id={file_id}" -O {zip_path}'
        )
        os.system(cmd)
        if (os.path.exists(zip_path)
                and os.path.getsize(zip_path) > 1e6):
            print("✓ wget OK")
            return True
    except Exception as e:
        print(f"  wget echec : {e}")

    # Méthode 4 : curl
    print("  Tentative curl...")
    try:
        os.system(
            f'curl -L -o {zip_path} '
            f'"https://drive.google.com/uc?export=download'
            f'&id={file_id}&confirm=t"'
        )
        if (os.path.exists(zip_path)
                and os.path.getsize(zip_path) > 1e6):
            print("✓ curl OK")
            return True
    except Exception as e:
        print(f"  curl echec : {e}")

    return False

if not os.path.exists(ZIP_PATH) or os.path.getsize(ZIP_PATH) < 1e4:
    ok = download_afnd(FILE_ID, ZIP_PATH)
    if not ok:
        raise RuntimeError(
            "❌ Téléchargement AFND impossible.\n"
            "Solutions :\n"
            "1. Vérifiez que le fichier Drive est public\n"
            "2. Téléchargez manuellement et uploadez dans Colab\n"
            "3. Montez Google Drive et ajustez le chemin"
        )
else:
    print(f"✓ ZIP déjà présent "
          f"({os.path.getsize(ZIP_PATH)/1e6:.1f} MB)")

print("Vérification ZIP...")
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        names = zf.namelist()
        top   = sorted(set(
            n.split('/')[0] for n in names if n.strip()
        ))
        print(f"  {len(names)} fichiers | structure : {top[:6]}")
except zipfile.BadZipFile:
    with open(ZIP_PATH, 'rb') as f:
        header = f.read(200)
    print(f"  Header : {header[:100]}")
    raise RuntimeError(
        "ZIP corrompu — probablement page HTML Google.\n"
        "→ Le fichier Drive n'est pas public, ou quota dépassé."
    )

already = [d for d in os.listdir(EXTRACT_TO)
           if os.path.isdir(os.path.join(EXTRACT_TO, d))]
if not already:
    print(f"Extraction → {EXTRACT_TO}...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_TO)
    print("✓ Extraction terminée.")
else:
    print(f"✓ Déjà extrait ({len(already)} éléments).")

# ── Détection automatique DATA_DIR / SOURCES_PATH ────────────
DATA_DIR     = None
SOURCES_PATH = None

for dirpath, dirnames, filenames in os.walk(EXTRACT_TO):
    if 'sources.json' in filenames:
        sp = os.path.join(dirpath, 'sources.json')
        for sub in dirnames:
            if sub.lower() == 'dataset':
                dd = os.path.join(dirpath, sub)
                if os.path.isdir(dd) and any(
                    s.startswith('source_')
                    for s in os.listdir(dd)
                ):
                    DATA_DIR     = dd
                    SOURCES_PATH = sp
                    break
        if DATA_DIR: break
        if any(d.startswith('source_') for d in dirnames):
            DATA_DIR     = dirpath
            SOURCES_PATH = sp
            break

if DATA_DIR is None:
    for dirpath, dirnames, filenames in os.walk(EXTRACT_TO):
        src_dirs = [d for d in dirnames
                    if d.startswith('source_')]
        if len(src_dirs) > 5:
            DATA_DIR = dirpath
            parent   = os.path.dirname(dirpath)
            for cand in [
                os.path.join(parent,  'sources.json'),
                os.path.join(dirpath, 'sources.json'),
            ]:
                if os.path.exists(cand):
                    SOURCES_PATH = cand
                    break
            break

if DATA_DIR is None:
    print("\n⚠ Arborescence extraite :")
    for dp, dn, fn in os.walk(EXTRACT_TO):
        lv = dp.replace(EXTRACT_TO, '').count(os.sep)
        if lv > 3: continue
        print('  '*lv + os.path.basename(dp)+'/')
        for f in fn[:3]: print('  '*(lv+1)+f)
    raise FileNotFoundError(
        "DATA_DIR introuvable. Vérifiez la structure du ZIP."
    )

n_src = len([d for d in os.listdir(DATA_DIR)
              if os.path.isdir(os.path.join(DATA_DIR, d))])

print(f"\n{'='*55}")
print(f"✓ AFND prêt")
print(f"  DATA_DIR     = {DATA_DIR}")
print(f"  SOURCES_PATH = {SOURCES_PATH}")
print(f"  Nb sources   = {n_src}")
print(f"{'='*55}")

# ── Constantes globales ───────────────────────────────────────
LABEL2ID = {'credible': 0, 'not credible': 1, 'undecided': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
FEAT_COLS = ['L_std', 'R_std', 'S_std']
SEEDS     = [0, 1, 2]

CHECKPOINTS = {
    'AraBERT'   : 'aubmindlab/bert-base-arabertv2',
    'MARBERT'   : 'UBC-NLP/MARBERTv2',
    'QARiB'     : 'moussaKam/bert-base-arabertv02-twitter',
    'CAMeLBERT' : 'CAMeL-Lab/bert-base-arabic-camelbert-msa',
    'XLM-R'     : 'xlm-roberta-base',
    'mBERT'     : 'bert-base-multilingual-cased',
}
SENTIMENT_CHECKPOINT = (
    'CAMeL-Lab/bert-base-arabic-camelbert-mix-sentiment'
)
DIALECT_MAPPING = {
    'MSA':'MSA','EGY':'Egyptian',
    'GLF':'Gulf','SAU':'Gulf','KUW':'Gulf',
    'UAE':'Gulf','QAT':'Gulf','BAH':'Gulf',
    'OMA':'Gulf','IRA':'Gulf',
    'LEV':'Levantine','SYR':'Levantine','LEB':'Levantine',
    'PAL':'Levantine','JOR':'Levantine',
    'MGH':'Maghrebi','MOR':'Maghrebi','ALG':'Maghrebi',
    'TUN':'Maghrebi','LIB':'Maghrebi',
    'SUD':'Other','YEM':'Other',
}
SENSATIONAL_WORDS = {
    'عاجل','خطير','صادم','مذهل','كارثة','فضيحة',
    'حصري','مرعب','لن تصدق','انفجار','انهيار',
    'مزيف','تضليل','كذب','خداع','مزور',
}
CREDIBILITY_WORDS = {
    'رسمي','موثق','مؤكد','تقرير','دراسة','بحث',
    'إحصاء','وزارة','حكومة','مصدر','وكالة','صحيفة',
    'أعلن','أكد','أفاد','نشر','وثّق','بيان','إعلان',
}
POSITIVE_WORDS_AR = {
    'جيد','ممتاز','رائع','إيجابي','نجاح','أمل','سعيد',
    'مبارك','تقدم','انتصار','فوز','خير','سلام','كريم',
    'عدل','إنجاز','ازدهار','صحيح','موثوق','حقيقي',
    'دقيق','مؤكد','رسمي','موثق',
}
NEGATIVE_WORDS_AR = {
    'سيء','خطأ','كذب','خطر','مشكلة','أزمة','فشل',
    'مزيف','خداع','تضليل','حرب','وفاة','إرهاب',
    'كارثة','فساد','سقوط','انهيار','مخيف','مريب',
    'مزور','مزعوم','مثير','صادم','مرعب','خطير',
    'مقلق','مدمر','زائف','مضلل','كاذب','شائعة','تحريض',
}
HEDGE_WORDS = {
    'ربما','قد','يُقال','يُزعم','يبدو','من المحتمل'
}
print("✓ Constantes définies.")

# ============================================================
# SECTION 3 — CHARGEMENT DATASET
# ============================================================
print("\nChargement AFND...")

with open(SOURCES_PATH, 'r', encoding='utf-8') as f:
    sources_data = json.load(f)

sources_df = pd.DataFrame([
    {'source': s, 'label': l}
    for s, l in sources_data.items()
])

def load_articles(fp):
    if os.path.exists(fp):
        with open(fp, 'r', encoding='utf-8') as f:
            return json.load(f).get('articles', [])
    return []

def get_all_articles(data_dir, sources):
    out = []
    for src in sources:
        fp   = os.path.join(data_dir, src, 'scraped_articles.json')
        arts = load_articles(fp)
        for a in arts:
            a['source'] = src
        out.extend(arts)
    return out

sources  = sources_df['source'].tolist()
articles = get_all_articles(DATA_DIR, sources)

articles_df = pd.DataFrame(articles)
df = pd.merge(articles_df, sources_df, how='inner', on='source')
df['label_id'] = df['label'].map(LABEL2ID)
df = df.dropna(subset=['label_id', 'text', 'title'])
df['label_id'] = df['label_id'].astype(int)

print(f"✓ {df.shape[0]:,} articles | {df.shape[1]} colonnes")
print(df['label'].value_counts())

colors = ['#2196F3', '#F44336', '#9E9E9E']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['label'].value_counts().plot.bar(
    ax=axes[0], color=colors, rot=30
)
axes[0].set_title('Distribution des labels')
axes[0].set_ylabel("Nombre d'articles")
df['label'].value_counts().plot.pie(
    ax=axes[1], colors=colors,
    autopct='%1.1f%%', startangle=90
)
axes[1].set_title('Proportions')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dataset_dist.pdf'))
plt.show()

# ============================================================
# SECTION 4 — PREPROCESSING ARABE
# ============================================================
def preprocess_text(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'@\w+',               '', text)
    text = re.sub(r'#\w+',               '', text)
    text = re.sub(r'http\S+',            '', text)
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)
    text = re.sub(r'[أإآ]',              'ا', text)
    text = re.sub(r'ؤ',                  'و', text)
    text = re.sub(r'[يى]',               'ي', text)
    text = re.sub(r'ة',                  'ه', text)
    text = re.sub(r'[\u064B-\u065F]',    '', text)
    text = re.sub(r'(.)\1{2,}',          r'\1\1', text)
    text = re.sub(r'\s+',                ' ', text).strip()
    return text

def remove_sw(text):
    return ' '.join(
        t for t in text.split()
        if t not in arabic_stopwords
    )

def full_preprocess(text):
    return remove_sw(preprocess_text(text))

df['full_text']  = (df['title'].fillna('') + ' '
                    + df['text'].fillna(''))
df['clean_text'] = df['full_text'].apply(full_preprocess)
print("✓ Preprocessing terminé.")
print(f"  Longueur moy. : "
      f"{df['clean_text'].apply(lambda x: len(x.split())).mean():.0f} mots")

# ============================================================
# SECTION 5 — SENTIMENT (corrigé — top_k=None)
# ============================================================
print(f"\nChargement sentiment : {SENTIMENT_CHECKPOINT}")

SENTIMENT_OK   = False
SENTIMENT_TOOL = None

try:
    # ✅ top_k=None → retourne les scores des 3 classes
    sentiment_pipe = hf_pipeline(
        'text-classification',
        model=SENTIMENT_CHECKPOINT,
        tokenizer=SENTIMENT_CHECKPOINT,
        device=0 if torch.cuda.is_available() else -1,
        truncation=True,
        max_length=512,
        top_k=None,
    )
    SENTIMENT_OK   = True
    SENTIMENT_TOOL = SENTIMENT_CHECKPOINT
    print("✓ CAMeLBERT-mix-sentiment chargé.")
except Exception as e:
    SENTIMENT_TOOL = 'Lexique ArSentiment (fallback)'
    print(f"  Fallback lexique : {e}")

def _sent_lex(text):
    toks = text.split()
    n    = max(len(toks), 1)
    p    = sum(1 for t in toks if t in POSITIVE_WORDS_AR)
    ng   = sum(1 for t in toks if t in NEGATIVE_WORDS_AR)
    return float(np.clip((p - ng) / n * 10, -1., 1.))

def compute_sentiment_batch(texts, bs=64):
    """
    Score continu ∈ [-1, 1] sans masse ponctuelle en 0.
    Utilise pos - neg sur les 3 classes retournées par top_k=None.
    """
    # ✅ Garde : si pipeline absent → fallback lexique
    if not SENTIMENT_OK:
        return [_sent_lex(t) for t in texts]

    scores = []
    tts    = [t[:1024] if isinstance(t, str) else ''
              for t in texts]

    for i in range(0, len(tts), bs):
        batch = tts[i:i+bs]
        try:
            rs = sentiment_pipe(batch)
            for r in rs:
                # r = [{'label':'positive','score':0.7},
                #      {'label':'negative','score':0.2},
                #      {'label':'neutral', 'score':0.1}]
                d   = {item['label'].lower(): item['score']
                       for item in r}
                pos = d.get('positive', d.get('pos', 0.))
                neg = d.get('negative', d.get('neg', 0.))
                # ✅ Score continu — plus de masse en 0.5
                w   = pos - neg
                scores.append(float(np.clip(w, -1., 1.)))
        except Exception:
            scores.extend([0.] * len(batch))
        if (i // bs) % 200 == 0:
            print(f"  Sentiment {i//bs+1}/{len(tts)//bs+1}...")
    return scores

# ── Calcul (avec cache) ───────────────────────────────────────
print("Calcul sentiment...")
cached_s = cache_load('sentiment_scores')
if cached_s is not None:
    raw_s = cached_s
else:
    if SENTIMENT_OK:
        raw_s = compute_sentiment_batch(df['clean_text'].tolist())
    else:
        raw_s = df['clean_text'].apply(_sent_lex).tolist()
    cache_save(raw_s, 'sentiment_scores')

df['S_raw_signed'] = raw_s
df['S_mapped']     = [(s + 1) / 2 for s in raw_s]

print(f"✓ Sentiment — outil : {SENTIMENT_TOOL}")
print(df['S_mapped'].describe().round(3))
# Après : pickle.dump(sentiment_scores, f)
# Ajouter :
drive_cache_file = Path("/content/drive/MyDrive/AFND_cache/sentiment_scores.pkl")
if drive_cache_file.parent.exists():
    shutil.copy(cache_file_local, drive_cache_file)
    print(f"  💾 Cache aussi sauvegardé sur Drive : {drive_cache_file}")

# Vérification anomalie
pct_05 = (df['S_mapped'] == 0.5).mean() * 100
print(f"  % S_mapped = 0.5 exactement : {pct_05:.1f}%")
if pct_05 > 20:
    print("  ⚠ Distribution encore biaisée — "
          "vérifier top_k=None dans le pipeline.")
else:
    print("  ✓ Distribution continue — OK.")

fig, ax = plt.subplots(figsize=(10, 4))
for lbl, c in zip(['credible', 'not credible', 'undecided'],
                   colors):
    df[df['label'] == lbl]['S_mapped'].hist(
        bins=30, alpha=0.5, label=lbl,
        density=True, ax=ax, color=c
    )
ax.set_xlabel('S_i ∈ [0,1]')
ax.set_ylabel('Densité')
ax.set_title('Distribution polarité S par classe')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sentiment_dist.pdf'))
plt.show()

# ============================================================
# SECTION 6 — FEATURES L, R, S
# ============================================================
def compute_L(text):
    return float(np.log1p(len(text.split())))

def build_freq(texts):
    c = Counter()
    for t in texts:
        c.update(t.split())
    return c

def compute_R(text, freq, tau):
    toks = text.split()
    if not toks: return 0.
    return sum(1 for w in toks
               if freq.get(w, 0) <= tau) / len(toks)

def add_lrs(df_in, freq=None, tau=None):
    d = df_in.copy()
    d['L_raw'] = d['clean_text'].apply(compute_L)

    # ✅ Condition robuste (tau=0 ou freq vide ne bloquent plus)
    if freq is not None and tau is not None:
        d['R_raw'] = d['clean_text'].apply(
            lambda t: compute_R(t, freq, tau)
        )
    else:
        d['R_raw'] = pd.Series([0.] * len(d), index=d.index)

    d['S_raw'] = d['S_mapped'].values
    return d

def standardize(tr, va, te,
                cols=('L_raw', 'R_raw', 'S_raw')):
    for c in cols:
        mu = tr[c].mean()
        sg = tr[c].std() + 1e-8
        nc = c.replace('_raw', '_std')
        for d in [tr, va, te]:
            d[nc] = (d[c] - mu) / sg
    return tr, va, te

def prepare_features(tr, va, te, tau_pct=0.30):
    """
    ✅ τ au 30ème percentile (vs 10ème) pour un signal R
    plus discriminant (évite de mesurer seulement les hapax).
    """
    freq = build_freq(tr['clean_text'].tolist())
    fv   = sorted(freq.values())
    tau  = fv[int(len(fv) * tau_pct)] if fv else 1
    tau  = max(tau, 2)      # au moins 2 pour dépasser les OOV purs
    print(f"  τ ({tau_pct*100:.0f}ème pct) = {tau}")
    tr = add_lrs(tr, freq, tau)
    va = add_lrs(va, freq, tau)
    te = add_lrs(te, freq, tau)
    tr, va, te = standardize(tr, va, te)
    return tr, va, te

# ============================================================
# SECTION 7 — SPLITS
# ============================================================
def dedup(df_tr):
    d    = df_tr.copy()
    d['tn'] = (d['title'].fillna('')
                .apply(preprocess_text)
                .str.lower())
    n    = len(d)
    d    = d.drop_duplicates(subset='tn')
    print(f"  Dédup : {n}→{len(d)} ({n-len(d)} doublons)")
    return d.drop(columns='tn').reset_index(drop=True)

def article_split(df, test=.2, val=.1, seed=0):
    tr_va, te = train_test_split(
        df, test_size=test,
        stratify=df['label_id'], random_state=seed
    )
    tr, va = train_test_split(
        tr_va, test_size=val/(1-test),
        stratify=tr_va['label_id'], random_state=seed
    )
    return (tr.reset_index(drop=True),
            va.reset_index(drop=True),
            te.reset_index(drop=True))

def source_disjoint_split(df, frac=.2, seed=0):
    rng = np.random.default_rng(seed)
    sl  = (df.groupby('source')['label_id']
             .agg(lambda x: x.mode()[0])
             .reset_index()
             .rename(columns={'label_id': 'dom'}))
    ts  = []
    for lbl in sl['dom'].unique():
        grp = sl[sl['dom'] == lbl]['source'].tolist()
        rng.shuffle(grp)
        ts.extend(grp[:max(1, int(len(grp) * frac))])
    ts_set = set(ts)
    te     = df[df['source'].isin(ts_set)]
    tr_va  = df[~df['source'].isin(ts_set)]
    tr, va = train_test_split(
        tr_va, test_size=.1,
        stratify=tr_va['label_id'], random_state=seed
    )
    ov = set(tr['source']) & set(te['source'])
    print(f"  Src-disjoint Tr:{len(tr)} Va:{len(va)} "
          f"Te:{len(te)} Overlap:{len(ov)}")
    return (tr.reset_index(drop=True),
            va.reset_index(drop=True),
            te.reset_index(drop=True))

print("\n=== Split article-level ===")
tr_art, va_art, te_art = article_split(df, seed=SEED)
tr_art = dedup(tr_art)
print(f"  Tr:{len(tr_art)} Va:{len(va_art)} Te:{len(te_art)}")

print("\n=== Split source-disjoint ===")
tr_src, va_src, te_src = source_disjoint_split(df, seed=SEED)
tr_src = dedup(tr_src)

print("\n=== Features article-level ===")
tr_art, va_art, te_art = prepare_features(
    tr_art, va_art, te_art, tau_pct=0.30
)

print("\n=== Features source-disjoint ===")
tr_src, va_src, te_src = prepare_features(
    tr_src, va_src, te_src, tau_pct=0.30
)

# ============================================================
# SECTION 8 — HEURISTIQUES (corrigée)
# ============================================================
def heur_score(text, L, R, S):
    toks  = text.split()
    n     = max(len(toks), 1)

    pct   = sum(1 for c in text
                if c in '!؟?…') / max(len(text), 1)
    sen   = sum(1 for t in toks
                if t in SENSATIONAL_WORDS) / n
    hdg   = sum(1 for t in toks
                if t in HEDGE_WORDS) / n
    cred  = sum(1 for t in toks
                if t in CREDIBILITY_WORDS) / n  # ← signal positif

    # Normalisation features standardisées → [0,1]
    L_norm = 1 / (1 + np.exp(-L))
    S_dist = abs(S - 0.5) * 2

    # Scores indépendants (pas de contrainte cr+nc+ud=1)
    score_nc = pct*2.0 + sen*3.0 + R*1.0 + S_dist*1.5
    score_ud = hdg*3.0 + (1 - L_norm)*1.5
    score_cr = cred*3.0 + L_norm*1.0 + (1 - S_dist)*0.5

    return np.array([score_cr, score_nc, score_ud])

def run_heuristics(df_te):
    preds = np.array([
        np.argmax(heur_score(
            r['clean_text'],
            r.get('L_std', 0),
            r.get('R_std', 0),
            r.get('S_std', 0)
        ))
        for _, r in df_te.iterrows()
    ])
    y   = df_te['label_id'].values
    f1  = f1_score(y, preds, average='macro', zero_division=0)
    acc = accuracy_score(y, preds)
    print(f"\n[Heuristics] F1={f1:.4f} Acc={acc:.4f}")
    print(classification_report(
        y, preds,
        target_names=['credible', 'not_credible', 'undecided']
    ))
    return preds, y

heur_preds, heur_labels = run_heuristics(te_art)

# ============================================================
# SECTION 9 — BASELINES N-GRAM
# ============================================================
from scipy.sparse import hstack, csr_matrix

baseline_results = {}

NGRAM_CLF_COMBOS = [
    ('lr',  (1, 1)),
    ('lr',  (1, 2)),
    ('lr',  (1, 3)),
    ('svm', (1, 1)),
    ('svm', (1, 2)),
    ('svm', (1, 3)),
    ('gb',  (1, 1)),
    ('gb',  (1, 2)),
    ('gb',  (1, 3)),
]

def run_ngram(tr, va, te, ngram=(1, 2),
              clf_name='lr', use_feat=True):
    vec = TfidfVectorizer(
        ngram_range=ngram,
        max_features=100_000,
        sublinear_tf=True
    )
    Xtr = vec.fit_transform(tr['clean_text'])
    Xva = vec.transform(va['clean_text'])
    Xte = vec.transform(te['clean_text'])

    if use_feat:
        Xtr = hstack([Xtr, csr_matrix(tr[FEAT_COLS].values)])
        Xva = hstack([Xva, csr_matrix(va[FEAT_COLS].values)])
        Xte = hstack([Xte, csr_matrix(te[FEAT_COLS].values)])

    ytr = tr['label_id'].values
    yva = va['label_id'].values
    yte = te['label_id'].values

    clfs = {
        'lr' : LogisticRegression(
                   max_iter=1000, class_weight='balanced',
                   C=1., random_state=SEED),
        'svm': LinearSVC(
                   max_iter=2000, class_weight='balanced',
                   C=1., random_state=SEED),
        'gb' : GradientBoostingClassifier(
                   n_estimators=200, max_depth=5,
                   random_state=SEED),
    }

    clf = clfs[clf_name]
    clf.fit(Xtr, ytr)
    tp  = clf.predict(Xte)
    tf1 = f1_score(yte, tp, average='macro', zero_division=0)
    ta  = accuracy_score(yte, tp)
    nm  = f"{clf_name}_ng{'_'.join(map(str, ngram))}"

    print(f"\n[{nm}] F1={tf1:.4f} Acc={ta:.4f}")
    print(classification_report(
        yte, tp,
        target_names=['credible', 'not_credible', 'undecided']
    ))
    return dict(
        model_name=nm, test_preds=tp,
        test_labels=yte, test_f1=tf1, test_acc=ta
    )

print("=== Baselines N-gram ===")
for clf_name, ng in NGRAM_CLF_COMBOS:
    k = f"{clf_name}_{'_'.join(map(str, ng))}"
    try:
        baseline_results[k] = run_ngram(
            tr_art, va_art, te_art,
            ngram=ng, clf_name=clf_name
        )
    except Exception as e:
        print(f"  ❌ {k} : {e}")

print("\n=== Récap Baselines ===")
bl_df = pd.DataFrame([
    {'Modèle'   : v['model_name'],
     'Macro-F1' : f"{v['test_f1']*100:.2f}%",
     'Accuracy' : f"{v['test_acc']*100:.2f}%"}
    for v in baseline_results.values()
]).sort_values('Macro-F1', ascending=False)
print(bl_df.to_string(index=False))
bl_df.to_csv(
    os.path.join(OUTPUT_DIR, 'baseline_results.csv'),
    index=False
)



Upgrade pip...

Installation des packages...
  ✓ transformers>=4.35.0
  ✓ datasets>=2.14.0
  ✓ tokenizers>=0.15.0
  ✓ sentencepiece>=0.1.99


KeyboardInterrupt: 

In [ ]:
# CELLULE URGENTE — Sauvegarde Drive dès que le cache est prêt
import time, shutil
from pathlib import Path

local_cache  = Path("/content/AFND_cache/sentiment_scores.pkl")
drive_cache  = Path("/content/drive/MyDrive/AFND_cache/sentiment_scores.pkl")
drive_cache.parent.mkdir(parents=True, exist_ok=True)

print("⏳ En attente du cache local...")
while not local_cache.exists():
    time.sleep(30)

shutil.copy(local_cache, drive_cache)
size = drive_cache.stat().st_size / 1e6
print(f"✅ Cache copié sur Drive ! ({size:.1f} MB)")
print(f"   → {drive_cache}")

In [ ]:
# ============================================================
# CELL 10 — Dataset class + TransformerFusion Architecture
# ============================================================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import numpy as np

# ── Constantes ──────────────────────────────────────────────
MAX_LEN      = 512
HEAD_TAIL    = True   # Head+Tail truncation
HEAD_TOKENS  = 128
TAIL_TOKENS  = 382    # HEAD + TAIL = 510 + 2 (CLS/SEP) = 512
LABEL_SMOOTH = 0.05
BATCH_SIZE   = 16
LR_TRANS     = 2e-5
NUM_EPOCHS   = 8
PATIENCE     = 2
NUM_LABELS   = 3
PROJ_DIM     = 32     # projection des features L,R,S
HIDDEN_DIM   = 128
IS_COLAB     = True
NUM_WORKERS  = 0 if IS_COLAB else 2

LABEL2ID = {"credible": 0, "not credible": 1, "undecided": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

TRANSFORMER_MODELS = {
    "AraBERT" : "aubmindlab/bert-base-arabertv2",
    "MARBERT" : "UBC-NLP/MARBERT",
    "QARiB"   : "qarib/bert-base-qarib",
    "CAMeLBERT": "CAMeL-Lab/bert-base-arabic-camelbert-mix",
    "XLM-R"   : "xlm-roberta-base",
    "mBERT"   : "bert-base-multilingual-cased",
}

# ── Dataset ─────────────────────────────────────────────────
class AFNDDataset(Dataset):
    def __init__(self, texts, feats, labels, tokenizer,
                 max_len=MAX_LEN, head_tail=HEAD_TAIL):
        self.texts     = texts
        self.feats     = feats          # numpy array (N, 3) → L, R, S
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.head_tail = head_tail

    def __len__(self):
        return len(self.texts)

    def _encode(self, text):
        tok = self.tokenizer
        if self.head_tail:
            ids = tok.encode(text, add_special_tokens=False,
                             max_length=10_000, truncation=True)
            if len(ids) > (self.max_len - 2):
                ids = ids[:HEAD_TOKENS] + ids[-(TAIL_TOKENS):]
            ids = [tok.cls_token_id] + ids + [tok.sep_token_id]
            mask = [1] * len(ids)
            pad_len = self.max_len - len(ids)
            ids  += [tok.pad_token_id] * pad_len
            mask += [0] * pad_len
        else:
            enc  = tok(text, max_length=self.max_len,
                       padding="max_length", truncation=True)
            ids  = enc["input_ids"]
            mask = enc["attention_mask"]
        return ids, mask

    def __getitem__(self, idx):
        ids, mask = self._encode(str(self.texts[idx]))
        return {
            "input_ids"      : torch.tensor(ids,  dtype=torch.long),
            "attention_mask" : torch.tensor(mask, dtype=torch.long),
            "feats"          : torch.tensor(self.feats[idx], dtype=torch.float),
            "labels"         : torch.tensor(self.labels[idx], dtype=torch.long),
        }

# ── TransformerFusion ────────────────────────────────────────
class TransformerFusion(nn.Module):
    """
    [CLS] → proj(L,R,S) → concat → Dense(128) → Softmax
    """
    def __init__(self, model_name, num_labels=NUM_LABELS,
                 feat_dim=3, proj_dim=PROJ_DIM, hidden_dim=HIDDEN_DIM,
                 label_smoothing=LABEL_SMOOTH):
        super().__init__()
        self.encoder   = AutoModel.from_pretrained(model_name)
        hidden_size    = self.encoder.config.hidden_size
        self.feat_proj = nn.Sequential(
            nn.Linear(feat_dim, proj_dim),
            nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + proj_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_labels),
        )
        self.loss_fn = nn.CrossEntropyLoss(
            label_smoothing=label_smoothing
        )

    def forward(self, input_ids, attention_mask, feats, labels=None):
        out  = self.encoder(input_ids=input_ids,
                            attention_mask=attention_mask)
        cls  = out.last_hidden_state[:, 0, :]   # [CLS]
        fp   = self.feat_proj(feats)             # (B, proj_dim)
        cat  = torch.cat([cls, fp], dim=-1)      # (B, hidden+proj)
        logits = self.classifier(cat)            # (B, num_labels)
        loss = self.loss_fn(logits, labels) if labels is not None else None
        return loss, logits

# ── Helpers ──────────────────────────────────────────────────
def get_feats(df):
    """Retourne numpy array (N,3) : L, R, S normalisés."""
    return df[["L", "R", "S_mapped"]].values.astype(np.float32)

def get_labels(df):
    return df["label"].map(LABEL2ID).values.astype(np.int64)

print("✅ Cell 10 — Dataset + TransformerFusion définis.")
print(f"   Modèles prévus : {list(TRANSFORMER_MODELS.keys())}")
print(f"   HEAD_TOKENS={HEAD_TOKENS} | TAIL_TOKENS={TAIL_TOKENS} | MAX_LEN={MAX_LEN}")
print(f"   BATCH_SIZE={BATCH_SIZE} | LR={LR_TRANS} | EPOCHS={NUM_EPOCHS} | PATIENCE={PATIENCE}")
print(f"   LABEL_SMOOTH={LABEL_SMOOTH} | PROJ_DIM={PROJ_DIM} | HIDDEN_DIM={HIDDEN_DIM}")
print(f"   NUM_WORKERS={NUM_WORKERS}")

✅ Cell 10 — Dataset + TransformerFusion définis.
   Modèles prévus : ['AraBERT', 'MARBERT', 'QARiB', 'CAMeLBERT', 'XLM-R', 'mBERT']
   HEAD_TOKENS=128 | TAIL_TOKENS=382 | MAX_LEN=512
   BATCH_SIZE=16 | LR=2e-05 | EPOCHS=8 | PATIENCE=2
   LABEL_SMOOTH=0.05 | PROJ_DIM=32 | HIDDEN_DIM=128
   NUM_WORKERS=0


In [ ]:
# ============================================================
# SECTION 11 — ARCHITECTURE FUSION
# ============================================================
class TransformerFusion(nn.Module):
    """[CLS] + proj 64-dim (L,R,S) → Dense(128) → Softmax."""

    def __init__(self, ckpt, n_cls=3, n_feat=3,
                 proj=64, drop=.1, eps=.05):
        super().__init__()
        self.enc  = AutoModel.from_pretrained(ckpt)
        hid       = self.enc.config.hidden_size
        self.proj = (nn.Sequential(
                         nn.Linear(n_feat, proj), nn.ReLU())
                     if n_feat > 0 else None)
        self.drop = nn.Dropout(drop)
        self.head = nn.Sequential(
            nn.Linear(hid + (proj if n_feat > 0 else 0), 128),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(128, n_cls)
        )
        self.eps   = eps
        self.n_cls = n_cls

    def forward(self, input_ids, attention_mask,
                token_type_ids=None, features=None,
                labels=None):
        kw = dict(input_ids=input_ids,
                  attention_mask=attention_mask)
        if token_type_ids is not None:
            kw['token_type_ids'] = token_type_ids

        out    = self.enc(**kw)
        cls    = self.drop(out.last_hidden_state[:, 0, :])
        z      = (torch.cat([cls, self.proj(features)], -1)
                  if self.proj and features is not None
                  else cls)
        logits = self.head(z)

        loss = None
        if labels is not None:
            ce   = nn.CrossEntropyLoss()(logits, labels)
            loss = ((1 - self.eps) * ce
                    + self.eps * (-torch.log(
                        torch.tensor(1. / self.n_cls))))
        return logits, loss

print("✓ TransformerFusion défini.")

# ============================================================
# SECTION 12 — ECE + TEMPERATURE SCALING
# ============================================================
def compute_ece(probs, labels, n_bins=15):
    """ECE — 15 bins equal-frequency."""
    conf = probs.max(1)
    pred = probs.argmax(1)
    corr = (pred == labels).astype(float)
    sidx = np.argsort(conf)
    conf = conf[sidx]
    corr = corr[sidx]
    n    = len(labels)
    ece  = 0.
    for b in range(n_bins):
        s = b * (n // n_bins)
        e = ((b + 1) * (n // n_bins)
             if b < n_bins - 1 else n)
        if e <= s:
            continue
        ece += ((e - s) / n
                * abs(corr[s:e].mean() - conf[s:e].mean()))
    return ece

class TempScaling(nn.Module):
    def __init__(self):
        super().__init__()
        self.T = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.T.clamp(min=.05)

    def fit(self, vl, vy, lr=.01, mi=500):
        self.train()
        opt = torch.optim.LBFGS([self.T], lr=lr, max_iter=mi)
        ce  = nn.CrossEntropyLoss()
        def step():
            opt.zero_grad()
            loss = ce(self(vl), vy)
            loss.backward()
            return loss
        opt.step(step)
        T = self.T.item()
        print(f"  T = {T:.4f}")
        self.eval()
        return T

def run_calibration(res, n_bins=15):
    nm = res['model_name']
    vl = res['val_logits_best']
    vy = res['val_labels_best']
    tl = torch.tensor(res['test_logits'], dtype=torch.float)
    ty = res['test_labels']

    pb = F.softmax(tl, -1).numpy()
    eb = compute_ece(pb, ty, n_bins)

    ts = TempScaling()
    T  = ts.fit(vl, vy)

    with torch.no_grad():
        pa = F.softmax(ts(tl), -1).numpy()
    ea = compute_ece(pa, ty, n_bins)

    print(f"[{nm}] ECE {eb*100:.2f}% → {ea*100:.2f}%  "
          f"Δ = {-(ea-eb)*100:.2f} pp")

    for ci, cn in ID2LABEL.items():
        m = ty == ci
        if m.sum() < n_bins:
            continue
        print(f"  ECE [{cn}] = "
              f"{compute_ece(pa[m], ty[m], n_bins)*100:.2f}%")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, pr, tt in zip(
        axes, [pb, pa],
        [f'Avant (ECE={eb*100:.1f}%)',
         f'Après T={T:.3f} (ECE={ea*100:.1f}%)']
    ):
        c  = pr.max(1)
        ok = (pr.argmax(1) == ty).astype(float)
        fp, mp = calibration_curve(
            ok, c, n_bins=n_bins, strategy='quantile'
        )
        ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Parfait')
        ax.plot(mp, fp, 'b-o', lw=2, ms=5, label=nm)
        ax.fill_between(mp, fp, mp, alpha=.15, color='blue')
        ax.set(
            xlabel='Confiance', ylabel='Frac. correcte',
            title=tt, xlim=[0, 1], ylim=[0, 1]
        )
        ax.legend()

    plt.suptitle(f'Calibration — {nm}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'calib_{nm}.pdf'))
    plt.show()

    return dict(T=T, ece_b=eb, ece_a=ea,
                pb=pb, pa=pa, ts=ts)

print("✓ ECE + TempScaling définis.")



✓ TransformerFusion défini.
✓ ECE + TempScaling définis.


In [ ]:
# ============================================================
# CELL 12 — Training loop (définition)
# ============================================================
import torch
import numpy as np
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score
import time, os, gc
from pathlib import Path

def train_transformer(
    model_name, train_df, val_df, test_df,
    batch_size=16, lr=2e-5,
    warmup=.1, use_feat=True,
    device=None                       # ← CORRIGÉ : None au lieu de DEVICE
):
    if device is None:                # ← résolu à l'appel, pas à la définition
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"\n{'='*55}")
    print(f"  Modèle : {model_name}")
    print(f"  Device : {device}")
    print(f"{'='*55}")

    # ── Tokenizer ──────────────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # ── Features & Labels ──────────────────────────────────
    tr_feats  = get_feats(train_df)
    va_feats  = get_feats(val_df)
    te_feats  = get_feats(test_df)
    tr_labels = get_labels(train_df)
    va_labels = get_labels(val_df)
    te_labels = get_labels(test_df)

    if not use_feat:
        tr_feats = np.zeros_like(tr_feats)
        va_feats = np.zeros_like(va_feats)
        te_feats = np.zeros_like(te_feats)

    # ── Datasets ───────────────────────────────────────────
    tr_texts = train_df["clean_text"].tolist()
    va_texts = val_df["clean_text"].tolist()
    te_texts = test_df["clean_text"].tolist()

    tr_ds = AFNDDataset(tr_texts, tr_feats, tr_labels, tokenizer)
    va_ds = AFNDDataset(va_texts, va_feats, va_labels, tokenizer)
    te_ds = AFNDDataset(te_texts, te_feats, te_labels, tokenizer)

    tr_dl = DataLoader(tr_ds, batch_size=batch_size,
                       shuffle=True,  num_workers=NUM_WORKERS)
    va_dl = DataLoader(va_ds, batch_size=batch_size*2,
                       shuffle=False, num_workers=NUM_WORKERS)
    te_dl = DataLoader(te_ds, batch_size=batch_size*2,
                       shuffle=False, num_workers=NUM_WORKERS)

    # ── Modèle ─────────────────────────────────────────────
    model = TransformerFusion(model_name).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr,
                                  weight_decay=1e-2)
    total_steps  = len(tr_dl) * NUM_EPOCHS
    warmup_steps = int(total_steps * warmup)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, warmup_steps, total_steps
    )

    # ── Early stopping ─────────────────────────────────────
    best_f1   = -1
    patience  = 0
    best_path = OUTPUT_DIR / f"{model_name.split('/')[-1]}_best.pt"

    # ── Boucle epochs ──────────────────────────────────────
    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        t0 = time.time()
        total_loss = 0

        for step, batch in enumerate(tr_dl):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            feats          = batch["feats"].to(device)
            labels         = batch["labels"].to(device)

            optimizer.zero_grad()
            loss, _ = model(input_ids, attention_mask, feats, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

            if (step + 1) % 500 == 0:
                print(f"    Ep{epoch} step {step+1}/{len(tr_dl)}"
                      f" — loss {total_loss/(step+1):.4f}")

        # ── Validation ─────────────────────────────────────
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in va_dl:
                input_ids      = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                feats          = batch["feats"].to(device)
                labels         = batch["labels"].to(device)
                _, logits = model(input_ids, attention_mask, feats)
                preds = logits.argmax(-1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.cpu().numpy())

        val_f1  = f1_score(all_labels, all_preds, average="macro")
        elapsed = (time.time() - t0) / 60
        print(f"  Epoch {epoch}/{NUM_EPOCHS} — "
              f"loss {total_loss/len(tr_dl):.4f} | "
              f"val_F1 {val_f1:.4f} | {elapsed:.1f} min")

        # ── Early stopping check ───────────────────────────
        if val_f1 > best_f1:
            best_f1   = val_f1
            patience  = 0
            torch.save(model.state_dict(), best_path)
            print(f"    ✅ Meilleur modèle sauvegardé (F1={best_f1:.4f})")
        else:
            patience += 1
            print(f"    ⏳ Patience {patience}/{PATIENCE}")
            if patience >= PATIENCE:
                print(f"    🛑 Early stopping à l'epoch {epoch}")
                break

    # ── Test avec le meilleur modèle ───────────────────────
    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for batch in te_dl:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            feats          = batch["feats"].to(device)
            labels         = batch["labels"].to(device)
            _, logits = model(input_ids, attention_mask, feats)
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            preds = probs.argmax(-1)
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    all_probs  = np.array(all_probs)
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    test_f1  = f1_score(all_labels, all_preds, average="macro")
    test_acc = accuracy_score(all_labels, all_preds)
    print(f"\n  🎯 Test — F1={test_f1:.4f} | Acc={test_acc:.4f}")

    # ── Nettoyage mémoire ──────────────────────────────────
    del model
    torch.cuda.empty_cache()
    gc.collect()

    return {
        "model_name" : model_name,
        "test_f1"    : test_f1,
        "test_acc"   : test_acc,
        "probs"      : all_probs,
        "preds"      : all_preds,
        "labels"     : all_labels,
        "best_path"  : str(best_path),
    }

print("✅ Cell 12 — train_transformer() défini.")
print(f"   Early stopping patience={PATIENCE} | Max epochs={NUM_EPOCHS}")
print(f"   Gradient clipping=1.0 | Weight decay=1e-2")
print(f"   Warmup=10% des steps | Scheduler=linear")

✅ Cell 12 — train_transformer() défini.
   Early stopping patience=2 | Max epochs=8
   Gradient clipping=1.0 | Weight decay=1e-2
   Warmup=10% des steps | Scheduler=linear


In [ ]:
# ============================================================
# CELL 13 — Entraînement des 6 Transformers
# ============================================================
from sklearn.metrics import classification_report
import json

transformer_results = {}

for model_key, model_name in TRANSFORMER_MODELS.items():
    print(f"\n{'#'*60}")
    print(f"# {model_key} — {model_name}")
    print(f"{'#'*60}")

    try:
        res = train_transformer(
            model_name  = model_name,
            train_df    = train_df,
            val_df      = val_df,
            test_df     = test_df,
            batch_size  = BATCH_SIZE,
            lr          = LR_TRANS,
            warmup      = 0.1,
            use_feat    = True,
        )
        transformer_results[model_key] = res

        # ── Rapport détaillé ───────────────────────────────
        target_names = [ID2LABEL[i] for i in range(NUM_LABELS)]
        print(classification_report(
            res["labels"], res["preds"],
            target_names=target_names, digits=4
        ))

        # ── Sauvegarde résultats partiels ──────────────────
        partial = {
            k: {
                "test_f1"  : v["test_f1"],
                "test_acc" : v["test_acc"],
            }
            for k, v in transformer_results.items()
        }
        with open(OUTPUT_DIR / "transformer_results_partial.json", "w") as f:
            json.dump(partial, f, indent=2)
        print(f"  💾 Résultats partiels sauvegardés.")

    except Exception as e:
        print(f"  ❌ Erreur {model_key} : {e}")
        transformer_results[model_key] = None
        torch.cuda.empty_cache()
        gc.collect()
        continue

# ── Récap final Cell 13 ────────────────────────────────────
print(f"\n{'='*60}")
print("  RÉCAP — 6 Transformers")
print(f"{'='*60}")
print(f"  {'Modèle':<15} {'F1 Macro':>10} {'Accuracy':>10}")
print(f"  {'-'*40}")
for k, v in transformer_results.items():
    if v is not None:
        print(f"  {k:<15} {v['test_f1']:>10.4f} {v['test_acc']:>10.4f}")
    else:
        print(f"  {k:<15} {'ERREUR':>10}")

# ── Sauvegarde finale ──────────────────────────────────────
final = {
    k: {"test_f1": v["test_f1"], "test_acc": v["test_acc"]}
    for k, v in transformer_results.items() if v is not None
}
with open(OUTPUT_DIR / "transformer_results_final.json", "w") as f:
    json.dump(final, f, indent=2)
print(f"\n  💾 Résultats finaux → {OUTPUT_DIR}/transformer_results_final.json")
print("✅ Cell 13 — Entraînement terminé.")


############################################################
# AraBERT — aubmindlab/bert-base-arabertv2
############################################################
  ❌ Erreur AraBERT : name 'train_df' is not defined

############################################################
# MARBERT — UBC-NLP/MARBERT
############################################################
  ❌ Erreur MARBERT : name 'train_df' is not defined

############################################################
# QARiB — qarib/bert-base-qarib
############################################################
  ❌ Erreur QARiB : name 'train_df' is not defined

############################################################
# CAMeLBERT — CAMeL-Lab/bert-base-arabic-camelbert-mix
############################################################
  ❌ Erreur CAMeLBERT : name 'train_df' is not defined

############################################################
# XLM-R — xlm-roberta-base
#####################################################

NameError: name 'OUTPUT_DIR' is not defined

In [ ]:
# ============================================================
# SECTION 15 — ABLATION 3 SEEDS
# ============================================================
ablation_configs = {
    'AraBERT+L+R+S' : FEAT_COLS,
    'No L (R+S)'    : ['R_std', 'S_std'],
    'No R (L+S)'    : ['L_std', 'S_std'],
    'No S (L+R)'    : ['L_std', 'R_std'],
    'No features'   : [],
}

ablation_results = {}

for cfg, fcols in ablation_configs.items():
    f1s, accs = [], []
    last_r    = None
    for sd in SEEDS:
        torch.manual_seed(sd)
        np.random.seed(sd)
        r = train_transformer(
            f"AraBERT_{cfg}_s{sd}",
            CHECKPOINTS['AraBERT'],
            tr_art, va_art, te_art,
            feat_cols=fcols,
            use_feat=len(fcols) > 0,
            max_epochs=8,
            patience=2
        )
        f1s.append(r['test_f1'])
        accs.append(r['test_acc'])
        last_r = r

    ablation_results[cfg] = dict(
        f1_mean     = np.mean(f1s),
        f1_std      = np.std(f1s),
        acc_mean    = np.mean(accs),
        acc_std     = np.std(accs),
        test_labels = last_r['test_labels'],
        test_preds  = last_r['test_preds'],
    )
    print(f"[Ablation] {cfg}: "
          f"F1={np.mean(f1s):.4f} ± {np.std(f1s):.4f}")

abl_df = pd.DataFrame([
    {
        'Variant'            : k,
        'Macro-F1 (mean±std)': f"{v['f1_mean']:.3f}±{v['f1_std']:.3f}",
        'Acc (mean±std)'     : f"{v['acc_mean']:.3f}±{v['acc_std']:.3f}",
    }
    for k, v in ablation_results.items()
])
print("\n=== Ablation Table ===")
print(abl_df.to_string(index=False))
abl_df.to_csv(
    os.path.join(OUTPUT_DIR, 'ablation_table.csv'),
    index=False
)

# Visualisation ablation
fig, ax = plt.subplots(figsize=(10, 5))
configs  = list(ablation_results.keys())
f1_means = [ablation_results[c]['f1_mean'] for c in configs]
f1_stds  = [ablation_results[c]['f1_std']  for c in configs]
ax.barh(configs, f1_means, xerr=f1_stds,
        color='steelblue', capsize=5, alpha=0.8)
ax.axvline(f1_means[0], color='red',
           linestyle='--', label='Full model', lw=1.5)
ax.set_xlabel('Macro-F1')
ax.set_title('Ablation — AraBERT + features (3 seeds)')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# SECTION 16 — BOOTSTRAP CIs
# ============================================================
def bootstrap_ci(yt, yp, metric='f1', n=1000, seed=0):
    rng = np.random.default_rng(seed)
    sc  = []
    N   = len(yt)
    for _ in range(n):
        idx = rng.integers(0, N, N)
        if metric == 'f1':
            sc.append(f1_score(yt[idx], yp[idx],
                               average='macro',
                               zero_division=0))
        else:
            sc.append(accuracy_score(yt[idx], yp[idx]))
    return (np.mean(sc),
            np.percentile(sc, 2.5),
            np.percentile(sc, 97.5))

ci_results = {}
print("=== Bootstrap 95% CIs ===")

for nm, res in transformer_results.items():
    mf, lb, ub = bootstrap_ci(
        res['test_labels'], res['test_preds'], 'f1'
    )
    ma, la, ua = bootstrap_ci(
        res['test_labels'], res['test_preds'], 'acc'
    )
    ci_results[nm] = dict(
        f1_mean=mf, f1_lb=lb, f1_ub=ub,
        acc_mean=ma, acc_lb=la, acc_ub=ua
    )
    print(f"  {nm:12s}  "
          f"F1={mf:.3f} [{lb:.3f}, {ub:.3f}]  "
          f"Acc={ma:.3f} [{la:.3f}, {ua:.3f}]")



In [ ]:
# ============================================================
# SECTION 17 — McNEMAR + HOLM-BONFERRONI
# ============================================================
if MLXTEND_OK and len(transformer_results) >= 2:
    comps = [
        ('AraBERT', 'MARBERT'),
        ('AraBERT', 'XLM-R'),
        ('AraBERT', 'mBERT'),
    ]
    print("=== McNemar Tests ===")
    pvs, cns = [], []

    for m1, m2 in comps:
        if (m1 not in transformer_results
                or m2 not in transformer_results):
            print(f"  ⚠ {m1} ou {m2} absent — skip")
            continue
        tb = mcnemar_table(
            transformer_results[m1]['test_labels'],
            transformer_results[m1]['test_preds'],
            transformer_results[m2]['test_preds']
        )
        chi2, p = mcnemar(tb, corrected=True)
        pvs.append(p)
        cns.append(f"{m1} vs {m2}")
        print(f"  {m1} vs {m2}  χ²={chi2:.4f}  p={p:.4f}")

    if pvs:
        print("\n[Holm-Bonferroni]")
        for rank, idx in enumerate(np.argsort(pvs)):
            aa = 0.05 / (len(pvs) - rank)
            print(f"  {cns[idx]}: p={pvs[idx]:.4f}  "
                  f"α_adj={aa:.4f}  → "
                  f"{'✓ sig' if pvs[idx] < aa else '✗ ns'}")
else:
    print("⚠ McNemar ignoré "
          "(mlxtend absent ou < 2 modèles).")



In [ ]:
============================================================
# SECTION 18 — CALIBRATION
# ============================================================
calibration_results = {}

for nm in ['AraBERT', 'MARBERT']:
    if nm in transformer_results:
        cal = run_calibration(
            transformer_results[nm], n_bins=15
        )
        calibration_results[nm] = cal
    else:
        print(f"  ⚠ {nm} absent — calibration ignorée.")

print("\n=== Récap calibration ===")
print(f"{'Model':12s}|{'T':6s}|"
      f"{'ECE avant':10s}|{'ECE après':10s}|{'Δ':8s}")
print('-' * 50)
for nm, c in calibration_results.items():
    d = (c['ece_b'] - c['ece_a']) * 100
    print(f"{nm:12s}|{c['T']:6.3f}|"
          f"{c['ece_b']*100:9.2f}%|"
          f"{c['ece_a']*100:9.2f}%|{d:+7.2f} pp")

# ============================================================
#

In [ ]:
# SECTION 19 — FAIRNESS DIALECTALE (corrigée)
# ============================================================
try:
    from camel_tools.dialect_identification import DialectIdentifier
    di = DialectIdentifier.pretrained()
    CAMEL_DI_OK = True
    print("✓ CAMeL Tools DI chargé.")
except Exception as e:
    CAMEL_DI_OK = False
    print(f"  CAMeL DI indisponible : {e}")

def identify_dialect(text, thr=.6):
    if not CAMEL_DI_OK:
        return 'Unknown', 0.
    try:
        pred = di.predict([text[:500]])[0]
        lbl  = pred.top
        sc   = pred.scores.get(lbl, 0.)
        if sc < thr:
            return 'Low_conf', sc
        return DIALECT_MAPPING.get(
            lbl[:3].upper(), 'Other'
        ), sc
    except Exception:
        return 'Unknown', 0.

def add_dialect(df_in, sample_n=None):
    d = df_in.copy()
    if sample_n:
        d = d.sample(
            n=min(sample_n, len(d)),
            random_state=SEED
        )
    dl, cf = [], []
    for txt in d['clean_text']:
        di_, c_ = identify_dialect(txt)
        dl.append(di_)
        cf.append(c_)
    d['dialect']      = dl
    d['dialect_conf'] = cf
    return d.reset_index(drop=True)

def fairness_analysis(df_te_d, preds):
    yt   = df_te_d['label_id'].values
    rows = []

    for dial in df_te_d['dialect'].unique():
        if dial in ('Unknown', 'Low_conf', 'Other'):
            continue
        msk = (df_te_d['dialect'] == dial).values
        if msk.sum() < 10:
            continue

        yd, pd_ = yt[msk],  preds[msk]
        yr, pr_ = yt[~msk], preds[~msk]

        top3 = (df_te_d[msk]['source']
                .value_counts(normalize=True)
                .head(3).sum() * 100)

        # ✅ EO gap corrigé — ignore classes absentes
        gaps = []
        for c in range(3):
            n_dial = (yd == c).sum()
            n_rest = (yr == c).sum()
            if n_dial == 0 or n_rest == 0:
                continue
            tpr_d = ((yd == c) & (pd_ == c)).sum() / n_dial
            tpr_r = ((yr == c) & (pr_ == c)).sum() / n_rest
            gaps.append(abs(tpr_d - tpr_r))

        eo_gap = (round(np.mean(gaps), 3)
                  if gaps else float('nan'))

        rows.append({
            'Dialect'      : dial,
            'N (test)'     : int(msk.sum()),
            '% test'       : f"{msk.sum()/len(yt)*100:.1f}%",
            'Src conc.(%)' : f"{top3:.1f}%",
            'Macro-F1'     : round(f1_score(
                                 yd, pd_, average='macro',
                                 zero_division=0), 3),
            'EO gap'       : eo_gap,
            'Accuracy'     : round(accuracy_score(yd, pd_), 3),
        })

    df_f = pd.DataFrame(rows).sort_values('Macro-F1')
    print("\n=== Fairness Analysis ===")
    print(df_f.to_string(index=False))
    return df_f

df_te_dial = add_dialect(te_art, sample_n=5000)
print(df_te_dial['dialect'].value_counts())

pf = (transformer_results['AraBERT']['test_preds'][:len(df_te_dial)]
      if 'AraBERT' in transformer_results
      else np.zeros(len(df_te_dial), dtype=int))

fair_df = fairness_analysis(df_te_dial, pf)
fair_df.to_csv(
    os.path.join(OUTPUT_DIR, 'fairness_table.csv'),
    index=False
)


In [ ]:

# ============================================================
# SECTION 20 — SOURCE-DISJOINT
# ============================================================
print("\n=== Source-Disjoint Evaluation (R5-C1) ===")

# ✅ dict global — remplace dir() et globals()
extra_results = {}

if 'AraBERT' in transformer_results:
    res_sd = train_transformer(
        'AraBERT_source_disjoint',
        CHECKPOINTS['AraBERT'],
        tr_src, va_src, te_src,
        feat_cols=FEAT_COLS,
        use_feat=True,
        max_epochs=8,
        patience=2
    )
    extra_results['res_sd'] = res_sd

    cmp_df = pd.DataFrame([
        {
            'Split'          : 'Random article-level (main)',
            'Accuracy'       : f"{transformer_results['AraBERT']['test_acc']*100:.1f}%",
            'Macro-F1'       : f"{transformer_results['AraBERT']['test_f1']*100:.1f}%",
            'Source overlap' : 'Yes',
            'N train src'    : tr_art['source'].nunique(),
            'N test src'     : te_art['source'].nunique(),
        },
        {
            'Split'          : 'Source-disjoint',
            'Accuracy'       : f"{res_sd['test_acc']*100:.1f}%",
            'Macro-F1'       : f"{res_sd['test_f1']*100:.1f}%",
            'Source overlap' : 'No',
            'N train src'    : tr_src['source'].nunique(),
            'N test src'     : te_src['source'].nunique(),
        },
    ])
    print(cmp_df.to_string(index=False))
    cmp_df.to_csv(
        os.path.join(OUTPUT_DIR,
                     'source_split_comparison.csv'),
        index=False
    )

    delta = ((transformer_results['AraBERT']['test_f1']
              - res_sd['test_f1']) * 100)
    print(f"\nΔ Macro-F1 = {delta:.2f} pp")
    if   abs(delta) < 2: print("→ Stable : cues article-level.")
    elif delta > 5:      print("→ Chute : signaux source détectés.")
    else:                print("→ Légère baisse modérée.")

# ============================================================
# SECTION 21 — OOD : politics+general → health
# ============================================================
print("\n=== OOD Evaluation (R5-C3) ===")

def infer_topic(src):
    sn = src.lower()
    if any(k in sn for k in
           ['health','saha','tibbi','طب','صح','medical']):
        return 'health'
    if any(k in sn for k in
           ['politic','siyasi','سياس','govern']):
        return 'politics'
    return 'general'

df['topic'] = df['source'].apply(infer_topic)
print(df['topic'].value_counts())

def make_ood_split(df, train_t=('politics', 'general'),
                    test_t='health', seed=0):
    trs = set(df[df['topic'].isin(train_t)]['source'])
    tes = set(df[df['topic'] == test_t]['source'])
    ov  = trs & tes
    print(f"  Overlap OOD : {len(ov)} src exclus")
    df_tr = df[df['topic'].isin(train_t)
               & ~df['source'].isin(ov)]
    df_te = df[(df['topic'] == test_t)
               & ~df['source'].isin(ov)]
    df_tr, df_va = train_test_split(
        df_tr, test_size=.1,
        stratify=df_tr['label_id'],
        random_state=seed
    )
    print(f"  OOD Tr:{len(df_tr)} "
          f"Va:{len(df_va)} Te:{len(df_te)}")
    return (df_tr.reset_index(drop=True),
            df_va.reset_index(drop=True),
            df_te.reset_index(drop=True))

def ood_weights(df_tr, df_va_h, gamma=.5):
    # ✅ fit sur train uniquement (pas de fuite test→train)
    vec = TfidfVectorizer(max_features=50_000)
    vec.fit(df_tr['clean_text'])
    Xt  = vec.transform(df_tr['clean_text']).toarray()
    Xh  = vec.transform(df_va_h['clean_text']).toarray()
    ctr = Xh.mean(0, keepdims=True)
    nt  = np.linalg.norm(Xt, axis=1, keepdims=True) + 1e-10
    nc  = np.linalg.norm(ctr) + 1e-10
    cos = (Xt @ ctr.T).squeeze(1) / (nt.squeeze(1) * nc)
    w   = np.exp(cos / gamma)
    return w / w.mean()

df_ood_tr, df_ood_va, df_ood_te = make_ood_split(df)
df_ood_tr, df_ood_va, df_ood_te = prepare_features(
    df_ood_tr, df_ood_va, df_ood_te, tau_pct=0.30
)

if 'AraBERT' in transformer_results:
    res_ood = train_transformer(
        'AraBERT_OOD_health',
        CHECKPOINTS['AraBERT'],
        df_ood_tr, df_ood_va, df_ood_te,
        feat_cols=FEAT_COLS,
        use_feat=True,
        max_epochs=8,
        patience=2
    )
    extra_results['res_ood'] = res_ood
    print(f"\n  OOD F1 health  : "
          f"{res_ood['test_f1']*100:.1f}%")
    print(f"  In-domain F1   : "
          f"{transformer_results['AraBERT']['test_f1']*100:.1f}%")


In [ ]:

# ============================================================
# SECTION 22 — MATRICES DE CONFUSION
# ============================================================
def plot_cm(yt, yp, name='', save=True):
    cm   = confusion_matrix(yt, yp)
    cmn  = cm.astype(float) / cm.sum(1, keepdims=True)
    lbls = ['Credible', 'Not Credible', 'Undecided']

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, data, fmt, sfx in zip(
        axes,
        [cm, cmn],
        ['d', '.2f'],
        ['(counts)', '(row-norm)']
    ):
        sns.heatmap(
            data, annot=True, fmt=fmt,
            xticklabels=lbls, yticklabels=lbls,
            cmap='Blues', ax=ax
        )
        ax.set_title(f"{name} {sfx}")
        ax.set_ylabel('True')
        ax.set_xlabel('Predicted')

    plt.tight_layout()
    if save:
        plt.savefig(
            os.path.join(OUTPUT_DIR, f'cm_{name}.pdf'),
            bbox_inches='tight'
        )
    plt.show()

for nm, res in transformer_results.items():
    plot_cm(res['test_labels'], res['test_preds'], nm)


In [ ]:

# ============================================================
# SECTION 23 — GRADIENT ATTRIBUTION (corrigée)
# ============================================================
def gradient_attr(model, tok, text,
                  feat_t=None, lbl=None,
                  device=DEVICE, topk=15):
    model.eval()
    enc = tok(
        text, return_tensors='pt',
        max_length=512, truncation=True,
        padding='max_length'
    ).to(device)

    emb_layer = model.enc.embeddings.word_embeddings
    # ✅ Pas de .detach() — retain_grad() conserve le gradient
    e = emb_layer(enc['input_ids'])
    e.retain_grad()

    out    = model.enc(
        inputs_embeds=e,
        attention_mask=enc['attention_mask']
    )
    cls    = out.last_hidden_state[:, 0, :]
    z      = (torch.cat([cls, model.proj(feat_t.to(device))], -1)
              if model.proj and feat_t is not None
              else cls)
    logits = model.head(z)

    if lbl is None:
        lbl = logits.argmax(-1).item()

    logits[0, lbl].backward()

    attrs  = (e.grad[0] * e[0]).norm(-1).detach().cpu().numpy()
    tokens = tok.convert_ids_to_tokens(
        enc['input_ids'][0].cpu().numpy()
    )

    SP    = {'[PAD]', '[SEP]', '[CLS]',
             '<pad>', '<s>', '</s>'}
    items = [{'tok': t, 'a': float(a)}
             for t, a in zip(tokens, attrs)
             if t not in SP]

    mx = max(i['a'] for i in items) if items else 1.
    for i in items:
        i['an'] = i['a'] / max(mx, 1e-10)

    return dict(
        tokens=items,
        pred=ID2LABEL[lbl],
        logits=logits.detach().cpu().numpy()
    )

def plot_attr(res, title='', topk=15, save=None):
    its = sorted(res['tokens'],
                 key=lambda x: x['an'],
                 reverse=True)[:topk]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(range(len(its)),
            [i['an'] for i in its],
            color='steelblue')
    ax.set_yticks(range(len(its)))
    ax.set_yticklabels(
        [i['tok'] for i in its], fontsize=10
    )
    ax.set_xlabel('Attribution normalisée')
    ax.set_title(f"{title}\nPréd : {res['pred']}")
    ax.invert_yaxis()

    plt.tight_layout()
    if save:
        plt.savefig(save, bbox_inches='tight')
    plt.show()

if 'AraBERT' in transformer_results:
    m_ab = transformer_results['AraBERT']['model']
    t_ab = transformer_results['AraBERT']['tokenizer']

    for tc in [0, 1, 2]:
        msk = te_art['label_id'] == tc
        if not msk.any():
            continue
        samp = te_art[msk].iloc[0]
        ft   = torch.tensor(
            samp[FEAT_COLS].values.astype(np.float32)
        ).unsqueeze(0)

        attr = gradient_attr(
            m_ab, t_ab,
            samp['full_text'],
            feat_t=ft, lbl=tc
        )
        cn = ID2LABEL[tc]
        plot_attr(
            attr,
            title=f"Vrai label : {cn}",
            save=os.path.join(
                OUTPUT_DIR, f'attr_{cn}.pdf'
            )
        )
        print(f"\nTop-10 [{cn}] :")
        for it in sorted(attr['tokens'],
                         key=lambda x: x['an'],
                         reverse=True)[:10]:
            print(f"  {it['tok']:20s} → {it['an']:.3f}")


In [ ]:

# ============================================================
# SECTION 24 — TABLE RÉSULTATS
# ============================================================
def build_table(tr_res, bl_res, ci_res):
    rows = []

    for nm, r in tr_res.items():
        ci = ci_res.get(nm, {})
        rows.append({
            'Model'    : nm,
            'Family'   : 'Transformer',
            'Acc'      : f"{r['test_acc']*100:.1f}%",
            'Macro-P'  : f"{precision_score(r['test_labels'], r['test_preds'], average='macro', zero_division=0)*100:.1f}%",
            'Macro-R'  : f"{recall_score(r['test_labels'], r['test_preds'], average='macro', zero_division=0)*100:.1f}%",
            'Macro-F1' : f"{r['test_f1']*100:.1f}%",
            '95% CI'   : f"[{ci.get('f1_lb',0)*100:.1f}%,"
                         f"{ci.get('f1_ub',0)*100:.1f}%]",
        })

    for nm, r in bl_res.items():
        rows.append({
            'Model'    : nm,
            'Family'   : 'N-gram',
            'Acc'      : f"{r['test_acc']*100:.1f}%",
            'Macro-P'  : '—',
            'Macro-R'  : '—',
            'Macro-F1' : f"{r['test_f1']*100:.1f}%",
            '95% CI'   : '—',
        })

    df_r = pd.DataFrame(rows)
    print("\n=== Results Table ===")
    print(df_r.to_string(index=False))
    return df_r

res_table = build_table(
    transformer_results, baseline_results, ci_results
)
res_table.to_csv(
    os.path.join(OUTPUT_DIR, 'main_results.csv'),
    index=False
)
